### В данном файле:
1) подключаемся через API (_и заранее скачанный API-токен_) к ресурсу, скачиваем и сразу распаковываем в папку /data
2) проверяем корректность подключения к БД, для чего выводим ее версию
3) хотим разбить датасет на две таблицы(чтобы в дальнейшем делать более интересные выборки), находящиеся в отношении *oneToOne*. Для этого создаем эти две таблицы, задав их структуру. Результат execute выводим в print чисто для знакомства с API, а также для того чтобы в дальнейшем(не в этом д/з) посмотреть разницу между psycopg2 и psycopg3
4) чисто для практики выполняем небольшую проверку - _выводим список таблиц_. Видим среди них только что созданные в п.3
5) для записи датасета в созданные ранее связанные таблицы используем (для знакомства) функционал _sqlalchemy_
6) поскольку данных много, выполняем проверку для случайного поля случайной записи. Для этого приложен скриншот pgAdmin`а

In [ ]:
%pip install kaggle

In [15]:
!kaggle datasets download -d miadul/introvert-extrovert-and-ambivert-classification -p ./data --unzip

Dataset URL: https://www.kaggle.com/datasets/miadul/introvert-extrovert-and-ambivert-classification
License(s): apache-2.0




  0%|          | 0.00/4.70M [00:00<?, ?B/s]
 21%|██▏       | 1.00M/4.70M [00:00<00:02, 1.57MB/s]
 43%|████▎     | 2.00M/4.70M [00:00<00:01, 2.72MB/s]
 64%|██████▍   | 3.00M/4.70M [00:00<00:00, 3.91MB/s]
 85%|████████▌ | 4.00M/4.70M [00:01<00:00, 5.15MB/s]
100%|██████████| 4.70M/4.70M [00:01<00:00, 4.18MB/s]


In [ ]:
%pip install psycopg2-binary

In [ ]:
%pip install sqlalchemy

In [5]:
from private.utils import cursor as cur

with cur() as cursor:
    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()
    print(f"PostgreSQL version: {db_version[0]}")


Successfully connected to the database!
PostgreSQL version: PostgreSQL 16.12, compiled by Visual C++ build 1944, 64-bit


In [3]:
# Создание таблицы "user_info" и "additional_user_info"
with cur() as cursor:    
    print(cursor.execute('''CREATE TABLE IF NOT EXISTS user_info (
        id INTEGER PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
        personality_type VARCHAR(50) NOT NULL,
        social_energy FLOAT NOT NULL,
        alone_time_preference FLOAT NOT NULL,
        talkativeness FLOAT NOT NULL,
        deep_reflection FLOAT NOT NULL,
        group_comfort FLOAT NOT NULL,
        party_liking FLOAT NOT NULL,
        listening_skill FLOAT NOT NULL,
        empathy FLOAT NOT NULL,
        creativity FLOAT NOT NULL,
        organization FLOAT NOT NULL,
        leadership FLOAT NOT NULL,
        risk_taking FLOAT NOT NULL,
        public_speaking_comfort FLOAT NOT NULL,
        curiosity FLOAT NOT NULL,
        routine_preference FLOAT NOT NULL,
        excitement_seeking FLOAT NOT NULL,
        friendliness FLOAT NOT NULL,
        emotional_stability FLOAT NOT NULL,
        planning FLOAT NOT NULL,
        spontaneity FLOAT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )'''))

    print(cursor.execute('''CREATE TABLE IF NOT EXISTS additional_user_info (
        user_id INTEGER PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
        adventurousness FLOAT NOT NULL,
        reading_habit FLOAT NOT NULL,
        sports_interest FLOAT NOT NULL,
        online_social_usage FLOAT NOT NULL,
        travel_desire FLOAT NOT NULL,
        gadget_usage FLOAT NOT NULL,
        work_style_collaborative FLOAT NOT NULL,
        decision_speed FLOAT NOT NULL,
        stress_handling FLOAT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        CONSTRAINT fk_user 
            FOREIGN KEY (user_id) 
            REFERENCES user_info(id) 
            ON DELETE CASCADE
    )'''))

Successfully connected to the database!
None
None


In [4]:
with cur() as cursor:    
    cursor.execute("""
        SELECT table_name FROM information_schema.tables 
        WHERE table_schema = 'public' AND table_type = 'BASE TABLE';
    """)
    tables = [row[0] for row in cursor.fetchall()]
    print(tables)

Successfully connected to the database!
['user_info', 'additional_user_info']


In [ ]:
%pip install sqlalchemy

In [6]:
from sqlalchemy import create_engine

from private.utils import get_url, prepare_personality_data

engine = create_engine(get_url())

df_source = prepare_personality_data()

df_table_1 = df_source[[
    "personality_type", 
    "social_energy", 
    "alone_time_preference",
    "talkativeness",
    "deep_reflection",
    "group_comfort",
    "party_liking",
    "listening_skill",
    "empathy",
    "creativity",
    "organization",
    "leadership",
    "risk_taking",
    "public_speaking_comfort",
    "curiosity",
    "routine_preference",
    "excitement_seeking",
    "friendliness",
    "emotional_stability",
    "planning",
    "spontaneity"
]].copy()
df_table_2 = df_source[[
    "adventurousness", 
    "reading_habit", 
    "sports_interest",
    "online_social_usage",
    "travel_desire",
    "gadget_usage",
    "work_style_collaborative",
    "decision_speed",
    "stress_handling"
]].copy()

# Write DataFrames to PostgreSQL
# 'if_exists="append"' adds rows. 'index=False' prevents pandas index from becoming a column.
try:
    df_table_1.to_sql(
        "user_info", con=engine, if_exists="append", index=False
    )
    df_table_2.to_sql(
        "additional_user_info", con=engine, if_exists="append", index=False
    )
    print("Database separation completed successfully!")
except Exception as e:
    print(f"An error occurred: {e}")

Database separation completed successfully!


P.S> Легкая проверка: на скриншоте ниже можно увидель, что во вторую таблицу в первую запись как раз попала информация из первой строки датасета (sports_interest=10,0)

![Скриншот БД для проверки](data/db_screen.png)